In [ ]:
import pandas as pd
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

import seaborn as sns

from sklearn.metrics import accuracy_score, recall_score, roc_auc_score, confusion_matrix, classification_report
df=pd.read_csv("full_data.csv")


In [ ]:
df.head()
# df.columns
# df.shape


In [ ]:
df.info()
df.isnull().sum()

نقوم بتحويل الاعمدة النصية والفئات الى ارقام:

In [ ]:
df['work_type'].unique()

In [ ]:
df['smoking_status'].unique()

In [ ]:
# تحويل الجنس
df['gender'] = df['gender'].map({'Male': 1, 'Female': 0})

# تحويل حالة الزواج
df['ever_married'] = df['ever_married'].map({'Yes': 1, 'No': 0})

# تحويل نوع السكن
df['Residence_type'] = df['Residence_type'].map({'Urban': 1, 'Rural': 0})

# إنشاء الكائن المحول
le_work = LabelEncoder()
le_smok = LabelEncoder()
# تطبيق الترميز على العمود work_type
df['work_type'] = le_work.fit_transform(df['work_type'])

# عرض القيم بعد الترميز
# print(dict(zip(le_work.classes_, le_work.transform(le_work.classes_))))
# print(df['work_type'].head())

# تطبيق الترميز على عمود التدخين
df['smoking_status'] = le_smok.fit_transform(df['smoking_status'])

# طباعة القيم بعد التحويل لمعرفة الخريطة
# print(dict(zip(le_smok.classes_, le_smok.transform(le_smok.classes_))))
# print(df[['smoking_status']].head())



نتاكد هل هناك توازن في البيانات 

In [ ]:
print(df["stroke"].value_counts())
category_counts = df["stroke"].value_counts()

import matplotlib.pyplot as plt

# رسم المخطط الدائري
plt.figure(figsize=(7,9))
plt.pie(category_counts,labels=category_counts.index,autopct='%1.1f%%',startangle=90,colors=plt.cm.Paired.colors)  # ألوان متعددة من colormap
plt.title("Distribution of Categories")
plt.show()

دراسة البيانات عن طريق الرسوم البيانية

In [ ]:
import matplotlib.pyplot as plt

for col in df.columns:
    plt.figure()
    if df[col].dtype == 'object' or df[col].nunique() < 20:
        df[col].value_counts().plot(kind='bar')
        plt.title(f"Distribution of {col}")
    else:
        df[col].hist(bins=30)
        plt.title(f"Distribution of {col}")
    plt.show()


كشف القيم الشاذه

In [ ]:
plt.figure(figsize=(12,6))
sns.boxplot(data=df.select_dtypes(include=['int64','float64']), orient="h")
plt.title("Boxplot لجميع الأعمدة الرقمية")
plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.boxplot(x=df['avg_glucose_level'])
plt.title("Boxplot of Average Glucose Level")
plt.show()


In [ ]:
# حساب الوسيط
median_glucose = df['avg_glucose_level'].median()

# استبدال القيم الكبيرة بالوسيط
df['avg_glucose_level'] = df['avg_glucose_level'].apply(
    lambda x: median_glucose if x > 200 else x
)

# التحقق من التغييرات
print(df[df['avg_glucose_level'] > 200])
print("Median used:", median_glucose)


الفرضيات: حيث نتحقق ما مدى تاثير كل حقل على النتيجة

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

target = 'stroke'

for col in df.columns:
    if col == target:
        continue  # نتجاوز العمود الهدف نفسه
    
    plt.figure(figsize=(6,4))
    
    # إذا العمود فئوي (object أو عدد فئات قليلة)
    if df[col].dtype == 'object' or df[col].nunique() <= 10:
        sns.countplot(x=col, hue=target, data=df)
        plt.title(f"Relationship between {col} and {target}")
        plt.ylabel("Count")
        plt.show()


بناء مودل تصنيفي باستخدام  خوارزمية RF

In [ ]:
num_cols = ['gender', 'age', 'hypertension', 'heart_disease', 'ever_married',
       'work_type', 'Residence_type', 'avg_glucose_level', 'bmi',
       'smoking_status']

X = df[num_cols]
y = df["stroke"]
from sklearn.preprocessing import RobustScaler

scaler = RobustScaler()
X_processed = scaler.fit_transform(X)

# ------------------------
# 5) تقسيم البيانات Train/Test
# ------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X_processed, y, test_size=0.2, random_state=42, stratify=y
)

# 6) إنشاء النماذج مع class_weight='balanced'
# ------------------------

model= RandomForestClassifier(class_weight='balanced', n_estimators=200, random_state=42,max_depth=18)


# ------------------------
# 7) تدريب وتقييم كل نموذج
# ------------------------
results = []


model.fit(X_train, y_train)
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:,1]  # احتمال الفئة 1
    
acc = accuracy_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
roc = roc_auc_score(y_test, y_prob)

print(f"Accuracy: {acc:.4f}")
print(f"Recall: {recall:.4f}")
print(f"ROC-AUC: {roc:.4f}")
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("Classification Report:")
print(classification_report(y_test, y_pred))

In [ ]:
print(len(model.estimators_))

In [ ]:
for i,tree in enumerate(model.estimators_):
    print(f"tree{i}depth:",tree.tree_.max_depth)

اختبار جميع الخوارزميات الاخرى

In [ ]:
num_cols = ['gender', 'age', 'hypertension', 'heart_disease', 'ever_married',
       'work_type', 'Residence_type', 'avg_glucose_level', 'bmi',
       'smoking_status']

X = df[num_cols]
y = df["stroke"]
from sklearn.preprocessing import RobustScaler

scaler = RobustScaler()
X_processed = scaler.fit_transform(X)

# ------------------------
# 5) تقسيم البيانات Train/Test
# ------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X_processed, y, test_size=0.2, random_state=42, stratify=y
)
# ------------------------
# 6) إنشاء النماذج مع class_weight='balanced'
# ------------------------
models = {
    'Decision Tree': DecisionTreeClassifier(class_weight='balanced', random_state=42),
    'Random Forest': RandomForestClassifier(class_weight='balanced', n_estimators=300, random_state=42),
    'Logistic Regression': LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42),
    "KNN": KNeighborsClassifier(n_neighbors=7),  # KNN لا يدعم class_weight
    "SVM": SVC(class_weight="balanced", kernel="rbf", probability=True, random_state=42)
}

# ------------------------
# 7) تدريب وتقييم كل نموذج
# ------------------------
results = []

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:,1]  # احتمال الفئة 1
    
    acc = accuracy_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    roc = roc_auc_score(y_test, y_prob)
   
    print(f"\n--- {name} ---")
    print(f"Accuracy: {acc:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"ROC-AUC: {roc:.4f}")
    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))
    print("Classification Report:")
    print(classification_report(y_test, y_pred))